# Counterfactual Explainer Benchmark

Benchmark counterfactual explainers across classifiers and datasets using `tscf_eval.benchmark`.

1. [Setup](#1.-Setup) — Imports, data loading, explainer and metric configuration
2. [Model Training](#2.-Model-Training) — HPO and training for all classifiers
3. [Single Benchmark](#3.-Single-Benchmark) — One model, one dataset
4. [Multi-Model Benchmark](#4.-Multi-Model-Benchmark) — All models, one dataset
5. [Multi-Dataset Benchmark](#5.-Multi-Dataset-Benchmark) — One model, all datasets
6. [Multi-Criteria Analysis](#6.-Multi-Criteria-Analysis) — Full metrics, Friedman, Pareto, consistency, LaTeX

## 1. Setup

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from aeon.classification.convolution_based import RocketClassifier
from aeon.classification.shapelet_based import RDSTClassifier
from aeon.classification.feature_based import Catch22Classifier
from aeon.classification.interval_based import TimeSeriesForestClassifier

from tscf_eval.data_loader import UCRLoader
from tscf_eval.benchmark import (
    BenchmarkRunner, BenchmarkResults, DatasetConfig, ModelConfig,
    ExplainerConfig, ParetoAnalyzer, WeightedScalarizer,
    friedman_test, format_latex_table,
)
from tscf_eval.counterfactuals import COMTE, NativeGuide, TSEvo, Glacier, LatentCF
from tscf_eval.evaluator import (
    Evaluator, Validity, Proximity, Sparsity, Plausibility,
    Diversity, Contiguity, Composition, Controllability, Confidence, Robustness, Efficiency,
)
from utils import evaluate_model, hpo_random_search, hpo_random_search_catch22
from utils import save_models, load_models

warnings.filterwarnings("ignore")

In [ ]:
# Global constants
DATASET_NAMES = ["ArrowHead", "Chinatown", "ECG200", "FaceFour", "SwedishLeaf"]
MODEL_TYPES = ["rocket", "rdst", "catch22_lr", "tsf"]
N_COUNTERFACTUALS = 3
N_INSTANCES = 12

### Load Datasets

In [ ]:
data = {}
for name in DATASET_NAMES:
    loader = UCRLoader(name)
    train = loader.load("train")
    test = loader.load("test")
    data[name] = {
        "X_train": np.asarray(train.X),
        "y_train": np.asarray(train.y),
        "X_test": np.asarray(test.X),
        "y_test": np.asarray(test.y),
    }

print("Datasets loaded:")
for name, d in data.items():
    print(f"  {name}: train {d['X_train'].shape}, test {d['X_test'].shape}")

### Configure Explainers

In [ ]:
explainer_configs = [
    # Nearest-neighbour based
    # ExplainerConfig("comte_euclidean", COMTE, {"distance": "euclidean"}, n_counterfactuals=N_COUNTERFACTUALS),
    ExplainerConfig("comte_dtw", COMTE, {"distance": "dtw"}, n_counterfactuals=N_COUNTERFACTUALS),
    # ExplainerConfig("ng_ng", NativeGuide, {"method": "ng"}, n_counterfactuals=N_COUNTERFACTUALS),
    ExplainerConfig("ng_dtw_dba", NativeGuide, {"method": "dtw_dba"}, n_counterfactuals=N_COUNTERFACTUALS),
    # Evolutionary
    ExplainerConfig("tsevo_authentic", TSEvo, {"transformer": "authentic", "n_generations": 10, "population_size": 20}, n_counterfactuals=N_COUNTERFACTUALS),
    # ExplainerConfig("tsevo_frequency", TSEvo, {"transformer": "frequency", "n_generations": 10, "population_size": 20}, n_counterfactuals=N_COUNTERFACTUALS),
    # ExplainerConfig("tsevo_all", TSEvo, {"transformer": "all", "n_generations": 10, "population_size": 20}, n_counterfactuals=N_COUNTERFACTUALS),
    # Gradient-based (require smooth decision surfaces)
    ExplainerConfig("glacier_uniform", Glacier, {"weight_type": "uniform", "max_iter": 50}, n_counterfactuals=N_COUNTERFACTUALS),
    # ExplainerConfig("glacier_local", Glacier, {"weight_type": "local", "max_iter": 50}, n_counterfactuals=N_COUNTERFACTUALS),
    ExplainerConfig("latentcf_uniform", LatentCF, {"step_weights": "uniform", "max_iter": 50}, n_counterfactuals=N_COUNTERFACTUALS),
    # ExplainerConfig("latentcf_local", LatentCF, {"step_weights": "local", "max_iter": 50}, n_counterfactuals=N_COUNTERFACTUALS),
]

print(f"{len(explainer_configs)} explainer variants configured")

### Configure Metrics

In [ ]:
evaluator = Evaluator([
    Validity(),
    Proximity(p=1), Proximity(p=2), Proximity(p=float("inf")),
    Sparsity(),
    Plausibility(method="lof"), Plausibility(method="if"), Plausibility(method="mp_ocsvm"),
    Diversity(),
    Contiguity(),
    Controllability(),
    Composition(),
    Confidence(),
    Robustness(),
    Efficiency(),
])

print(f"{len(evaluator.metrics)} metrics: {[m.name() for m in evaluator.metrics]}")

## 2. Model Training

Train all classifier types on all datasets using random-search HPO (optimizing F1 macro).

| Model | HPO parameter | Internal estimator | Smooth gradients? |
|-------|--------------|-------------------|-------------------|
| ROCKET | `n_kernels` | RidgeClassifierCV | Yes |
| RDST | `max_shapelets` | RidgeClassifierCV | Yes |
| Catch22 + LR | regularization `C` | LogisticRegression | Yes |
| TSF | `n_estimators`, `min_interval_length` | — | No (tree-based) |

In [ ]:
# HPO parameter distributions
ROCKET_PARAM_DIST = {"n_kernels": [500, 1000, 2000, 3000, 5000, 7500, 10000]}
RDST_PARAM_DIST = {"max_shapelets": [5, 10, 15, 20, 30]}
CATCH22_LR_PARAM_DIST = {
    "estimator": [
        LogisticRegression(C=c, max_iter=1000, random_state=42)
        for c in [0.01, 0.1, 1.0, 10.0]
    ],
}
TSF_PARAM_DIST = {
    "n_estimators": [50, 100, 150, 200, 250, 300],
    "min_interval_length": [5, 10, 15, 20],
}

N_ITER = 10  # random search iterations

In [ ]:
TSC_MODEL_METRICS = [
    ("accuracy", {}),
    ("precision", {"average": "macro"}),
    ("recall", {"average": "macro"}),
    ("f1", {"average": "macro"}),
]

models = {mt: {} for mt in MODEL_TYPES}
hpo_results = {mt: {} for mt in MODEL_TYPES}

print(f"Training {len(MODEL_TYPES)} model types x {len(DATASET_NAMES)} datasets (n_iter={N_ITER})")
print("=" * 100)

for dataset_name in DATASET_NAMES:
    d = data[dataset_name]
    cv_folds = 2

    print(f"\n{dataset_name} (n_train={len(d['X_train'])}, cv={cv_folds}):")
    print(f"{'Model':<12} {'Best Params':<50} {'CV F1':>10} {'Test F1':>10}")
    print("-" * 84)

    # ROCKET
    m, p, s = hpo_random_search(RocketClassifier, ROCKET_PARAM_DIST, d["X_train"], d["y_train"], n_iter=N_ITER, cv=cv_folds)
    models["rocket"][dataset_name], hpo_results["rocket"][dataset_name] = m, {"params": p, "cv_score": s}
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'rocket':<12} {str(p):<50} {s:>10.2%} {f1:>10.2%}")

    # RDST
    m, p, s = hpo_random_search(RDSTClassifier, RDST_PARAM_DIST, d["X_train"], d["y_train"], n_iter=N_ITER, cv=cv_folds)
    models["rdst"][dataset_name], hpo_results["rdst"][dataset_name] = m, {"params": p, "cv_score": s}
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'rdst':<12} {str(p):<50} {s:>10.2%} {f1:>10.2%}")

    # Catch22 + LogisticRegression
    m, p, s = hpo_random_search_catch22(Catch22Classifier, CATCH22_LR_PARAM_DIST, d["X_train"], d["y_train"], n_iter=N_ITER, cv=cv_folds)
    models["catch22_lr"][dataset_name], hpo_results["catch22_lr"][dataset_name] = m, {"params": p, "cv_score": s}
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'catch22_lr':<12} {str(p):<50} {s:>10.2%} {f1:>10.2%}")

    # TSF
    m, p, s = hpo_random_search(TimeSeriesForestClassifier, TSF_PARAM_DIST, d["X_train"], d["y_train"], n_iter=N_ITER, cv=cv_folds)
    models["tsf"][dataset_name], hpo_results["tsf"][dataset_name] = m, {"params": p, "cv_score": s}
    f1 = evaluate_model(m, d["X_test"], d["y_test"], TSC_MODEL_METRICS)["f1"]
    print(f"{'tsf':<12} {str(p):<50} {s:>10.2%} {f1:>10.2%}")

print(f"\nTrained {len(MODEL_TYPES) * len(DATASET_NAMES)} models")

In [ ]:
# HPO summary table
summary_data = []
for mt in MODEL_TYPES:
    for ds in DATASET_NAMES:
        r = hpo_results[mt][ds]
        y_pred = models[mt][ds].predict(data[ds]["X_test"])
        test_f1 = f1_score(data[ds]["y_test"], y_pred, average="macro")
        summary_data.append({
            "Model": mt, "Dataset": ds,
            "Best Params": str(r["params"]),
            "CV F1": f"{r['cv_score']:.2%}",
            "Test F1": f"{test_f1:.2%}",
        })

pd.DataFrame(summary_data)

In [ ]:
# Save trained models (or load pre-trained ones)
# save_models(models, hpo_results, MODEL_TYPES, DATASET_NAMES)

# To load pre-trained models instead of training:
models, hpo_results = load_models(MODEL_TYPES, DATASET_NAMES)

## 3. Single Benchmark

Compare all explainers using **one classifier on one dataset**.
Uses RDST on ArrowHead as a representative example.

In [ ]:
SINGLE_DATASET = "ArrowHead"
SINGLE_MODEL = "rdst"

d = data[SINGLE_DATASET]

single_results = BenchmarkRunner(
    datasets=[DatasetConfig(SINGLE_DATASET, d["X_train"], d["y_train"], d["X_test"], d["y_test"])],
    models=[ModelConfig(SINGLE_MODEL, models[SINGLE_MODEL][SINGLE_DATASET])],
    explainers=explainer_configs,
    evaluator=evaluator,
    instance_selection="stratified_confidence",
    n_instances=N_INSTANCES,
    n_jobs=-1,
    verbose=True,
).run()

with open(f"single_results_stratified_{SINGLE_MODEL}_{SINGLE_DATASET}.json", "w") as f:
    json.dump(single_results.to_dict(), f, indent=2, default=str)
print(f"Saved single_results_stratified_{SINGLE_MODEL}_{SINGLE_DATASET}.json")

In [ ]:
single_results.to_dataframe()

## 4. Multi-Model Benchmark

Compare explainers across **all classifiers on one dataset**.
This reveals how classifier choice affects counterfactual quality.

In [ ]:
MM_DATASET = "Chinatown"
d = data[MM_DATASET]

mm_dataset_config = DatasetConfig(MM_DATASET, d["X_train"], d["y_train"], d["X_test"], d["y_test"])
mm_model_configs = [ModelConfig(mt, models[mt][MM_DATASET]) for mt in MODEL_TYPES]

multi_model_results = BenchmarkRunner(
    datasets=[mm_dataset_config],
    models=mm_model_configs,
    explainers=explainer_configs,
    evaluator=evaluator,
    instance_selection="stratified_confidence",
    n_instances=N_INSTANCES,
    n_jobs=-1,
    verbose=True,
).run()

with open(f"multi_model_results_stratified_{MM_DATASET}.json", "w") as f:
    json.dump(multi_model_results.to_dict(), f, indent=2, default=str)
print(f"Saved multi_model_results_stratified_{MM_DATASET}.json")

In [ ]:
# Filter results for a specific model
SELECTED_MODEL = "rocket"  # Change to: "rocket", "rdst", "catch22_lr", "catch22_rf", "tsf"

print(f"Results for model: {SELECTED_MODEL}")
multi_model_results.filter(models=[SELECTED_MODEL]).to_dataframe()

In [ ]:
# Pivot table: compare one metric across all models
METRIC_TO_COMPARE = "validity"

df = multi_model_results.to_dataframe()
df.pivot(index="explainer", columns="model", values=METRIC_TO_COMPARE)

In [ ]:
# Aggregated statistics
print("By model:")
display(multi_model_results.aggregate(by="model"))

print("\nBy explainer:")
display(multi_model_results.aggregate(by="explainer"))

## 5. Multi-Dataset Benchmark

Compare explainers across **all datasets using one classifier per dataset**.
This identifies methods that perform consistently across different data characteristics.

In [ ]:
MD_MODEL = "rocket"

md_dataset_configs = [
    DatasetConfig(ds, data[ds]["X_train"], data[ds]["y_train"], data[ds]["X_test"], data[ds]["y_test"])
    for ds in DATASET_NAMES
]
md_model_configs = [ModelConfig(MD_MODEL, models[MD_MODEL][ds]) for ds in DATASET_NAMES]

# Run per dataset (each has its own fitted model)
multi_dataset_results = BenchmarkResults()
for ds_config, m_config in zip(md_dataset_configs, md_model_configs):
    print(f"Running {ds_config.name}...")
    for result in BenchmarkRunner(
        datasets=[ds_config], models=[m_config],
        explainers=explainer_configs,
        evaluator=evaluator,
        instance_selection="stratified_confidence",
        n_instances=N_INSTANCES,
        n_jobs=-1, verbose=False,
    ).run():
        multi_dataset_results.add(result)

print(f"\n{len(multi_dataset_results)} results collected")

with open(f"multi_dataset_results_stratified_{MD_MODEL}.json", "w") as f:
    json.dump(multi_dataset_results.to_dict(), f, indent=2, default=str)
print(f"Saved multi_dataset_results_stratified_{MD_MODEL}.json")

In [ ]:
# Filter by dataset
SELECTED_DATASET = "ArrowHead"  # Change to any dataset

print(f"Results for {SELECTED_DATASET}:")
multi_dataset_results.filter(datasets=[SELECTED_DATASET]).to_dataframe()

In [ ]:
print("By dataset:")
display(multi_dataset_results.aggregate(by="dataset"))

print("\nBy explainer (averaged across datasets):")
display(multi_dataset_results.aggregate(by="explainer"))

## 6. Multi-Criteria Analysis

When optimizing multiple metrics simultaneously, no single method is best at everything.
This section organizes results for the paper:

- **6.1 Full metrics table** — All 13 metrics aggregated by explainer (paper Table)
- **6.2 Friedman test** — Statistical significance per metric across datasets
- **6.3 Pareto dominance** — Per-category Pareto analysis (core, distribution, model, performance)
- **6.4 Pareto front plots** — 2D trade-off visualizations
- **6.5 Cross-dataset consistency** — Which methods are consistently Pareto-optimal
- **6.6 Weighted scalarization** — Composite ranking and sensitivity analysis
- **6.7 LaTeX export** — Publication-ready tables

In [ ]:
# All metrics from the evaluator
ALL_METRICS = [
    "validity",
    "proximity_linf",
    "sparsity",
    "plausibility_if",
    "diversity_dpp",
    "contiguity",
    "mean_n_segments",
    "mean_avg_segment_len",
    "mean_conf_orig",
    "mean_conf_cf",
    "robustness_lipschitz",
    "efficiency_time_s",
]

# Metrics grouped by the quality dimensions from the paper (Table 1)
METRIC_CATEGORIES = {
    "Core Quality":       ["validity", "proximity_linf", "sparsity"],
    "Distribution":       ["plausibility_if", "diversity_dpp"],
    "Structure":          ["contiguity", "mean_n_segments"],
    "Model Behavior":     ["mean_conf_cf"],
    "Stability":          ["robustness_lipschitz"],
    "Performance":        ["efficiency_time_s"],
}

# Representative subset for Pareto analysis (one per category, avoids redundancy)
PARETO_METRICS = [
    "validity", "proximity_linf", "sparsity",
    "plausibility_if", "diversity_dpp",
    "contiguity", "mean_n_segments",
    "mean_conf_cf",
    "robustness_lipschitz",
    "efficiency_time_s",
]

print(f"Total metrics: {len(ALL_METRICS)}")
print(f"Pareto subset: {len(PARETO_METRICS)}")
for cat, metrics in METRIC_CATEGORIES.items():
    print(f"  {cat}: {metrics}")

### 6.1 Full Metrics Table (Paper Table)

All 13 metrics aggregated by explainer across datasets. This becomes the main results table in the paper.

In [ ]:
# Full aggregated table — all metrics, averaged across datasets
agg_df = multi_dataset_results.aggregate(by="explainer")
cols_to_show = ["explainer"] + [m for m in ALL_METRICS if m in agg_df.columns]
display(agg_df[cols_to_show])

# Per-category breakdown
for cat, metrics in METRIC_CATEGORIES.items():
    available = [m for m in metrics if m in agg_df.columns]
    if available:
        print(f"\n{cat}:")
        display(agg_df[["explainer"] + available])

### 6.2 Friedman Statistical Test

Non-parametric Friedman test per metric across datasets. Tests whether explainer differences are statistically significant (p < 0.05).

In [ ]:
# Friedman test for ALL metrics across datasets
friedman_summary = []
for metric in ALL_METRICS:
    try:
        fr = friedman_test(multi_dataset_results, metric=metric)
        sig = "***" if fr.p_value < 0.001 else "**" if fr.p_value < 0.01 else "*" if fr.p_value < 0.05 else "n.s."
        friedman_summary.append({
            "metric": metric,
            "chi2": f"{fr.statistic:.2f}",
            "p_value": f"{fr.p_value:.4f}",
            "significant": sig,
            "best (rank 1)": fr.rankings.index[0],
        })
    except ValueError as e:
        friedman_summary.append({"metric": metric, "chi2": "-", "p_value": "-", "significant": str(e), "best (rank 1)": "-"})

friedman_df = pd.DataFrame(friedman_summary)
display(friedman_df)

### 6.3 Pareto Dominance Analysis

Using the representative subset (7 metrics, one per category) to avoid over-saturation of the Pareto front. With 13 metrics, almost every method would be non-dominated, making the analysis uninformative.

In [ ]:
# Pareto analysis with representative subset
analyzer = ParetoAnalyzer(metrics=PARETO_METRICS)

# Pareto front on single benchmark (illustrative)
pareto_methods = analyzer.pareto_front(single_results)
print(f"Pareto-optimal ({len(pareto_methods)}/{len(explainer_configs)}):")
for m in pareto_methods:
    print(f"  - {m}")

# Full dominance ranking on multi-dataset results
print(f"\nDominance ranking (multi-dataset, {len(PARETO_METRICS)} metrics):")
ranking = analyzer.dominance_ranking(multi_dataset_results)
display(ranking)

### 6.4 Pareto Front Plots

2D projections of the Pareto front showing key trade-offs between metrics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

analyzer.plot_front(
    multi_dataset_results, x_metric="validity", y_metric="proximity_l2",
    title="Validity vs Proximity", ax=axes[0],
)
analyzer.plot_front(
    multi_dataset_results, x_metric="validity", y_metric="sparsity",
    title="Validity vs Sparsity", ax=axes[1],
)
analyzer.plot_front(
    multi_dataset_results, x_metric="validity", y_metric="efficiency_time_s",
    title="Validity vs Efficiency", ax=axes[2],
)

plt.tight_layout()
plt.show()

### 6.5 Cross-Dataset Pareto Consistency

Which methods are Pareto-optimal across multiple datasets? Methods appearing on more fronts are more robust.

In [ ]:
results_by_dataset = {
    ds: multi_dataset_results.filter(datasets=[ds])
    for ds in DATASET_NAMES
}

consistency_df = analyzer.consistency(results_by_dataset)
display(consistency_df)

ax = analyzer.plot_consistency_heatmap(consistency_df)
plt.tight_layout()
plt.show()

### 6.6 Weighted Scalarization

Combine all metrics into a single composite score. Useful for producing a final ranking when a single "winner" is needed.

In [ ]:
# Equal-weight composite across ALL metrics
scalarizer = WeightedScalarizer(metrics=ALL_METRICS)
scores = scalarizer.score(multi_dataset_results)
display(scores)

# Sensitivity: how does ranking change as validity weight increases?
sens_df = scalarizer.sensitivity(multi_dataset_results, vary_metric="validity", n_steps=11)
ax = scalarizer.plot_sensitivity(sens_df, title="Sensitivity to validity weight")
plt.tight_layout()
plt.show()

### 6.7 LaTeX Export

Publication-ready tables with best-value highlighting and directional arrows.

In [ ]:
# Full aggregated metrics table (main results table for the paper)
agg_df = multi_dataset_results.aggregate(by="explainer")
cols_to_show = ["explainer"] + [m for m in ALL_METRICS if m in agg_df.columns]
print("=" * 40)
print("Table: Full aggregated metrics")
print("=" * 40)
print(format_latex_table(
    agg_df[cols_to_show],
    caption="Aggregated counterfactual evaluation metrics by explainer across all datasets (ROCKET classifier). "
            "Best values are bold. Arrows indicate optimization direction.",
    label="tab:full_metrics",
    midrule_every=4,
))

In [ ]:
# Pareto dominance ranking table
print("=" * 40)
print("Table: Dominance ranking")
print("=" * 40)
print(analyzer.to_latex(
    multi_dataset_results,
    caption="Pareto dominance ranking across datasets (7 representative metrics).",
    label="tab:dominance_ranking",
))

In [ ]:
# Friedman test summary table
print("=" * 40)
print("Table: Friedman test summary")
print("=" * 40)
print(format_latex_table(
    friedman_df,
    bold_best=False,
    arrows=False,
    caption="Friedman test results for each metric across datasets. "
            "Significance levels: *** p<0.001, ** p<0.01, * p<0.05, n.s. not significant.",
    label="tab:friedman_test",
))